In [1]:
# ============================================================================
# BƯỚC 1: SETUP MÔI TRƯỜNG
# ============================================================================
print("="*70)
print("BƯỚC 1: Cài đặt môi trường")
print("="*70)

# Kiểm tra GPU
!nvidia-smi

# Cài đặt ultralytics
!pip install ultralytics -q

print("\n✓ Đã cài đặt ultralytics")

# ============================================================================
# BƯỚC 2: MOUNT GOOGLE DRIVE (nếu dataset ở Drive)
# ============================================================================
print("\n" + "="*70)
print("BƯỚC 2: Mount Google Drive")
print("="*70)

from google.colab import drive
drive.mount('/content/drive')

print("✓ Đã mount Google Drive")


BƯỚC 1: Cài đặt môi trường
Mon Dec 22 11:13:09 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------

In [2]:
# ============================================================================
# BƯỚC 3: UPLOAD DATASET
# ============================================================================
print("\n" + "="*70)
print("BƯỚC 3: Chuẩn bị dataset")
print("="*70)

import os
import yaml

# ===== CHỌN 1 TRONG 2 CÁCH =====


# Uncomment và sửa đường dẫn
DATASET_PATH = '/content/drive/MyDrive/GroupProject_Seg'  # THAY ĐỔI PATH NÀY


print(f"Dataset path: {DATASET_PATH}")

# Kiểm tra cấu trúc dataset
print("\nKiểm tra cấu trúc dataset:")
for split in ['train', 'val']:
    img_dir = os.path.join(DATASET_PATH, 'images', split)
    lbl_dir = os.path.join(DATASET_PATH, 'labels', split)

    if os.path.exists(img_dir):
        n_imgs = len([f for f in os.listdir(img_dir)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f"✓ {split}/images: {n_imgs} ảnh")
    else:
        print(f"✗ Không tìm thấy: {img_dir}")

    if os.path.exists(lbl_dir):
        n_lbls = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])
        print(f"✓ {split}/labels: {n_lbls} file")
    else:
        print(f"✗ Không tìm thấy: {lbl_dir}")


BƯỚC 3: Chuẩn bị dataset
Dataset path: /content/drive/MyDrive/GroupProject_Seg

Kiểm tra cấu trúc dataset:
✓ train/images: 5964 ảnh
✓ train/labels: 5964 file
✓ val/images: 1486 ảnh
✓ val/labels: 1486 file


In [3]:
# BƯỚC 4: TẠO FILE DATA.YAML CHO COLAB
# ============================================================================
print("\n" + "="*70)
print("BƯỚC 4: Tạo data.yaml")
print("="*70)

# ⚠️⚠️⚠️ THAY ĐỔI CÁC GIÁ TRỊ THEO DATASET CỦA BẠN ⚠️⚠️⚠️
data_yaml_content = {
    'path': DATASET_PATH,              # Đường dẫn dataset
    'train': 'images/train',           # Đường dẫn tương đối đến train images
    'val': 'images/val',               # Đường dẫn tương đối đến val images

    'nc': 4,                           # ⚠️ SỐ LƯỢNG CLASSES
    'names': ['Stairs', 'crosswalk','sidewalk', 'tree-lined' ]      # ⚠️ TÊN CÁC CLASSES
}

# Lưu file yaml
yaml_path = '/content/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

print(f"✓ Đã tạo: {yaml_path}")
print("\nNội dung data.yaml:")
with open(yaml_path, 'r') as f:
    print(f.read())


BƯỚC 4: Tạo data.yaml
✓ Đã tạo: /content/data.yaml

Nội dung data.yaml:
names:
- Stairs
- crosswalk
- sidewalk
- tree-lined
nc: 4
path: /content/drive/MyDrive/GroupProject_Seg
train: images/train
val: images/val



In [ ]:
from pathlib import Path
import shutil

def clamp01(x: float) -> float:
    return max(0.0, min(1.0, x))

def bbox_to_rect_poly_line(parts):
    # parts: [cls, cx, cy, w, h] normalized
    cls = parts[0]
    cx, cy, w, h = map(float, parts[1:5])
    x1, y1 = clamp01(cx - w/2), clamp01(cy - h/2)
    x2, y2 = clamp01(cx + w/2), clamp01(cy + h/2)
    poly = [x1, y1,  x2, y1,  x2, y2,  x1, y2]  # 4 points
    return cls + " " + " ".join(f"{p:.6f}" for p in poly)

def convert_labels_dir(lbl_dir: Path):
    changed_files = 0
    changed_lines = 0

    for txt in lbl_dir.rglob("*.txt"):
        lines = [l.strip() for l in txt.read_text().splitlines() if l.strip()]
        if not lines:
            continue

        new_lines = []
        changed = False

        for line in lines:
            parts = line.split()
            if len(parts) == 5:  # bbox-only -> convert
                new_lines.append(bbox_to_rect_poly_line(parts))
                changed = True
                changed_lines += 1
            else:
                new_lines.append(line)

        if changed:
            shutil.copy2(txt, str(txt) + ".bak")  # backup
            txt.write_text("\n".join(new_lines) + "\n")
            changed_files += 1

    return changed_files, changed_lines

for split in ["train", "val"]:
    d = Path(DATASET_PATH) / "labels" / split
    if d.exists():
        f, l = convert_labels_dir(d)
        print(f"[{split}] converted files: {f}, converted bbox-lines: {l}")
    else:
        print(f"[{split}] not found:", d)


[train] converted files: 0, converted bbox-lines: 0
[val] converted files: 0, converted bbox-lines: 0


In [ ]:
from pathlib import Path

# 1) THƯ MỤC LABEL TRAIN CỦA BẠN
labels_train_dir = Path("/content/drive/MyDrive/GroupProject_Seg/labels/val")
print("Thư mục labels train:", labels_train_dir)
print("Tồn tại không? ", labels_train_dir.exists())

# 2) PHÂN LOẠI FILE TXT
detect_files = []   # chỉ bbox (5 cột)
seg_files = []      # segmentation (>=7 cột, (n-5) chẵn)
mixed_files = []    # 1 file có cả 5 cột lẫn seg
empty_files = []    # file rỗng
broken_files = []   # số cột linh tinh

for txt in labels_train_dir.rglob("*.txt"):
    with open(txt, "r") as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]

    if len(lines) == 0:
        empty_files.append(str(txt))
        continue

    has_5 = False
    has_seg = False
    bad_line = False

    for line in lines:
        parts = line.split()
        n = len(parts)

        if n == 5:
            has_5 = True           # dạng bbox
        elif n > 5 and (n - 5) % 2 == 0:
            has_seg = True         # dạng seg (có polygon)
        else:
            bad_line = True        # số cột kì lạ
            break

    if bad_line:
        broken_files.append(str(txt))
    elif has_5 and not has_seg:
        detect_files.append(str(txt))
    elif has_seg and not has_5:
        seg_files.append(str(txt))
    elif has_5 and has_seg:
        mixed_files.append(str(txt))

print("\n===== FILE DETECTION (chỉ bbox – 5 cột) =====")
print("Tổng:", len(detect_files))
for p in detect_files[:50]:
    print(p)

print("\n===== FILE SEGMENTATION (có polygon) =====")
print("Tổng:", len(seg_files))
for p in seg_files[:50]:
    print(p)

print("\n===== FILE MIXED (vừa 5 cột vừa seg) =====")
print("Tổng:", len(mixed_files))
for p in mixed_files[:50]:
    print(p)

print("\n===== FILE LABEL RỖNG =====")
print("Tổng:", len(empty_files))
for p in empty_files[:50]:
    print(p)

print("\n===== FILE FORMAT LỖI (số cột linh tinh) =====")
print("Tổng:", len(broken_files))
for p in broken_files[:50]:
    print(p)


Thư mục labels train: /content/drive/MyDrive/GroupProject_Seg/labels/val
Tồn tại không?  True

===== FILE DETECTION (chỉ bbox – 5 cột) =====
Tổng: 0

===== FILE SEGMENTATION (có polygon) =====
Tổng: 1486
/content/drive/MyDrive/GroupProject_Seg/labels/val/ds1__ds1__ds1__IMG_5866_frame_02220_502895_jpg.rf.7cc721028bfde504db39c556b747e721.txt
/content/drive/MyDrive/GroupProject_Seg/labels/val/ds1__ds1__ds1__IMG_5868_frame_00990_b1113f_jpg.rf.5d9fcfc08fc4ae4aa5c9a471d54d56da.txt
/content/drive/MyDrive/GroupProject_Seg/labels/val/ds1__ds1__ds1__IMG_5864_frame_00000_9522d8_jpg.rf.c9dbd9cb3cd76b635ae12ed0d0012283.txt
/content/drive/MyDrive/GroupProject_Seg/labels/val/ds1__ds1__ds1__IMG_8234_frame_00750_f53165_jpg.rf.12b667f1e0e5687bc39b4cbafcd1b660.txt
/content/drive/MyDrive/GroupProject_Seg/labels/val/ds1__ds1__ds1__IMG_5867_frame_00540_d46c72_jpg.rf.8d1caafa353c3d390ddb5f95056c18dd.txt
/content/drive/MyDrive/GroupProject_Seg/labels/val/ds1__ds1__ds1__IMG_8227_frame_01560_9000cf_jpg.rf.ae8fb

In [ ]:
# ============================================================================
# BƯỚC 5: LOAD MÔ HÌNH YOLOV8X
# ============================================================================
print("\n" + "="*70)
print("BƯỚC 5: Load YOLOv8x-")
print("="*70)

from ultralytics import YOLO

# Dùng model segmentation, không phải bản detection
model = YOLO('yolov8x-seg.pt')  # hoặc 'yolov8x-seg.pt' nếu bạn muốn bản x
print("✓ Đã load YOLOv8x-seg (segmentation pretrained model)")


import os
from ultralytics import YOLO

# ====== LƯU RUN NGAY TRONG FOLDER DATASET ======
RUNS_DIR  = os.path.join(DATASET_PATH, "runs")
EXP_NAME  = "yolov8x_seg_finetune"

WEIGHTS_DIR = os.path.join(RUNS_DIR, EXP_NAME, "weights")
LAST_CKPT   = os.path.join(WEIGHTS_DIR, "last.pt")
BEST_CKPT   = os.path.join(WEIGHTS_DIR, "best.pt")

# ====== CHỌN KIỂU LƯU ======
# False = tiết kiệm (chỉ last.pt/best.pt, last ghi đè mỗi epoch)
# True  = lưu thêm epoch*.pt mỗi epoch (tốn dung lượng)
SAVE_EVERY_EPOCH = False

# ====== AUTO RESUME NẾU ĐÃ CÓ CHECKPOINT ======
if os.path.exists(LAST_CKPT):
    print(f"✅ Found checkpoint -> RESUME from: {LAST_CKPT}")
    model = YOLO(LAST_CKPT)
    results = model.train(resume=True)
else:
    print("🆕 No checkpoint -> START from yolov8m-seg.pt")
    model = YOLO("yolov8m-seg.pt")

    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=640,
        batch=32,
        device=0,

        project=RUNS_DIR,   # ✅ trong folder dataset
        name=EXP_NAME,
        exist_ok=True,

        cache=True,
        workers=8,

        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=0.0,
        translate=0.1,
        scale=0.5,
        shear=0.0,
        perspective=0.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.0,
        copy_paste=0.0,

        patience=50,
        save=True,
        save_period=(1 if SAVE_EVERY_EPOCH else 0),  # 1=checkpoint mỗi epoch, 0=tiết kiệm

        optimizer='auto',
        lr0=0.01,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3.0,
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,

        box=7.5,
        cls=0.5,
        dfl=1.5,

        val=True,
        plots=True,
        verbose=True,
        seed=0,
        rect=False,
        cos_lr=False,
        close_mosaic=10,
        amp=True,
        fraction=1.0,
    )

print("\n📌 Weights saved at:", WEIGHTS_DIR)
print("   last exists?", os.path.exists(LAST_CKPT))
print("   best exists?", os.path.exists(BEST_CKPT))



BƯỚC 5: Load YOLOv8x-
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ Đã load YOLOv8x-seg (segmentation pretrained model)
✅ Found checkpoint -> RESUME from: /content/drive/MyDrive/GroupProject_Seg/runs/yolov8x_seg_finetune/weights/last.pt
Ultralytics 8.3.240 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasi

In [ ]:
#============================================================================
# BƯỚC 7+8: ĐÁNH GIÁ BẰNG BIỂU ĐỒ (learning curves + val plots)
#============================================================================
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

print("\n" + "="*70)
print("BƯỚC 7+8: Evaluation bằng biểu đồ")
print("="*70)

EXP_DIR = os.path.join(DATASET_PATH, "runs", "yolov8m_seg_finetune")
best_model_path = os.path.join(EXP_DIR, "weights", "best.pt")
results_csv = os.path.join(EXP_DIR, "results.csv")

print("EXP_DIR       :", EXP_DIR)
print("best_model    :", best_model_path, "exists?", os.path.exists(best_model_path))
print("results.csv   :", results_csv, "exists?", os.path.exists(results_csv))

# --------------------------
# (A) VẼ LEARNING CURVES TỪ results.csv
# --------------------------
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]

    def plot_cols(cols, title, ylabel):
        cols = [c for c in cols if c in df.columns]
        if not cols:
            print(f"⚠️ Không tìm thấy cột cho: {title}")
            return
        plt.figure()
        for c in cols:
            plt.plot(df[c].values, label=c)
        plt.title(title)
        plt.xlabel("Epoch")
        plt.ylabel(ylabel)
        plt.legend()
        plt.grid(True)
        plt.show()

    # Loss curves (tuỳ version có thể thiếu/đổi tên cột, nên chọn linh hoạt)
    loss_cols = [c for c in df.columns if ("loss" in c.lower()) and (c.startswith("train/") or c.startswith("val/"))]
    plot_cols(loss_cols, "Loss curves (train/val)", "Loss")

    # Metrics cho MASK (segmentation) thường có (M)
    mask_metric_cols = [c for c in df.columns if c.startswith("metrics/") and "(M)" in c]
    plot_cols(mask_metric_cols, "Mask metrics curves (Precision/Recall/mAP)", "Score")

    # (Tuỳ chọn) Metrics cho BOX thường có (B)
    box_metric_cols = [c for c in df.columns if c.startswith("metrics/") and "(B)" in c]
    if box_metric_cols:
        plot_cols(box_metric_cols, "Box metrics curves (Precision/Recall/mAP)", "Score")
else:
    print("⚠️ Không có results.csv để vẽ learning curves.")

# --------------------------
# (B) CHẠY VAL VÀ HIỂN THỊ PLOTS CHUẨN (PR/F1/Confusion…)
# --------------------------
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)

    EVAL_DIR = os.path.join(EXP_DIR, "eval_plots")  # lưu luôn trong folder dataset/runs/...
    metrics = model.val(
        data=yaml_path,
        imgsz=640,
        device=0,
        project=EVAL_DIR,
        name="val_best",
        exist_ok=True,
        plots=True
    )

    # In nhanh vài chỉ số chính (mask)
    if hasattr(metrics, "seg") and hasattr(metrics.seg, "map50"):
        print("\n📊 Mask (Seg) metrics:")
        print(f"  seg mAP50:     {metrics.seg.map50:.4f}")
        print(f"  seg mAP50-95:  {metrics.seg.map:.4f}")
        print(f"  seg Precision: {metrics.seg.mp:.4f}")
        print(f"  seg Recall:    {metrics.seg.mr:.4f}")

    # Hiển thị các ảnh plot mà Ultralytics xuất ra
    out_dir = os.path.join(EVAL_DIR, "val_best")
    print("\n📁 Val plots saved at:", out_dir)

    plot_imgs = []
    for ext in ("*.png", "*.jpg"):
        plot_imgs += glob.glob(os.path.join(out_dir, ext))
    plot_imgs = sorted(plot_imgs)

    # Lọc các plot “hay dùng” nếu có
    preferred = ["confusion_matrix", "PR_curve", "F1_curve", "P_curve", "R_curve", "results"]
    preferred_imgs = []
    for key in preferred:
        preferred_imgs += [p for p in plot_imgs if key in os.path.basename(p)]

    show_list = preferred_imgs if preferred_imgs else plot_imgs
    if not show_list:
        print("⚠️ Không thấy file plot image nào trong thư mục val.")
    else:
        for p in show_list[:10]:  # giới hạn hiển thị 10 ảnh
            print("Showing:", os.path.basename(p))
            img = Image.open(p)
            plt.figure()
            plt.imshow(img)
            plt.axis("off")
            plt.title(os.path.basename(p))
            plt.show()
else:
    print("⚠️ Chưa có best.pt để đánh giá.")
